In [ ]:
!pip install docling rapidocr-onnxruntime opencv-python-headless


In [ ]:
!pip install llama-index llama-index-llms-ollama pandas

In [ ]:
!pip install llama-index-llms-openai-like

In [ ]:
import os
from pathlib import Path
import cv2
import tempfile
import time
import json
from typing import List
import pandas as pd
from google.colab.patches import cv2_imshow

from llama_index.llms.ollama import Ollama
import socket
from llama_index.llms.openai_like import OpenAILike
from google.colab import userdata

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, ImageFormatOption
from docling_core.types.doc import ImageRefMode

In [ ]:
def preprocess_image(image_path):

    im = cv2.imread(image_path)
    gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

    # Повышение резкости
    sharpen = cv2.GaussianBlur(gray, (0, 0), 3)
    sharpen = cv2.addWeighted(gray, 1.5, sharpen, -0.5, 0)

    # Применение адаптивной пороговой фильтрации
    thresh = cv2.adaptiveThreshold(sharpen, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 21, 15)

    # Создание временного файла для сохранения предобработанного изображения
    temp_dir = tempfile.gettempdir()
    preprocessed_path = os.path.join(temp_dir, f"preprocessed_{Path(image_path).name}")

    cv2.imwrite(preprocessed_path, thresh)

    return preprocessed_path

In [ ]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True

pipeline_options.do_formula_enrichment = True       # Распознавание формул
pipeline_options.generate_picture_images = True     # Извлечение картинок

# Настройка RapidOcr (PaddleOCR)
ocr_options = RapidOcrOptions()
pipeline_options.ocr_options = ocr_options

# Применяем конфигурацию к поддерживаемым форматам
format_options = {
    InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    InputFormat.IMAGE: ImageFormatOption(pipeline_options=pipeline_options)
}

converter = DocumentConverter(format_options=format_options)

In [ ]:
def run_ocr_pipeline(file_path, converter, output_dir, Assets = False):
    """
    Принимает файл, при необходимости делает препроцессинг, запускает Docling. Если передан Assets = True, сохраняет картинки в отдельную папку и возвращает текст с ссылками на изображения.
    """
    path = Path(file_path)
    suffix = path.suffix.lower()
    is_image = suffix in [".png", ".jpg", ".jpeg"]
    target_path = str(path)
    temp_file_to_clean = None

    if is_image:
        target_path = preprocess_image(str(path))
        temp_file_to_clean = target_path

    result = converter.convert(target_path)

    if temp_file_to_clean and os.path.exists(temp_file_to_clean):
        os.remove(temp_file_to_clean)

    doc_dict = result.document.export_to_dict()

    if Assets == True:
        # Создаем папку для изображений текущего документа
        assets_dir = output_dir / f"{path.stem}_assets"
        assets_dir.mkdir(parents=True, exist_ok=True)
        md_path = output_dir / f"{path.stem}.md"

        # Сохраняем Markdown с внешними ссылками на картинки
        result.document.save_as_markdown(
            filename=md_path,
            image_mode=ImageRefMode.REFERENCED,
            artifacts_dir=assets_dir
        )

        # Читаем файл, чтобы вернуть текст для вывода в консоль
        with open(md_path, "r", encoding="utf-8") as f:
            markdown_text = f.read()
    else:
        # Если Assets = False, картинки сохраняются в base64
        markdown_text = result.document.export_to_markdown(image_mode=ImageRefMode.EMBEDDED)

    return markdown_text, doc_dict

In [ ]:
input_folder = Path("/content/drive/MyDrive/ocr_samples")
allowed_extensions = {'.pdf', '.jpg', '.jpeg', '.png'}

output_folder = Path("/content/drive/MyDrive/md_results2")
output_folder.mkdir(parents=True, exist_ok=True)

files_to_process = [f for f in input_folder.iterdir() if f.is_file() and f.suffix.lower() in allowed_extensions]
print(f"Найдено файлов для обработки: {len(files_to_process)}")

for file_path in files_to_process:
    print(f"\nОбработка файла: {file_path.name}")
    print("-" * 60)
    start_time = time.time()

    # Assets=True - сохраняем Markdown файл и картинки в отдельную папку
    md_result, doc_dict = run_ocr_pipeline(str(file_path), converter, output_dir=output_folder, Assets=True)

    processing_time = time.time() - start_time
    minutes = int(processing_time // 60)
    seconds  = processing_time % 60

    json_output_path = output_folder / (file_path.stem + ".json")

    # Сохраняем JSON файл
    with open(json_output_path, "w", encoding="utf-8") as f:
        json.dump(doc_dict, f, ensure_ascii=False, indent=2)

    print(f"Время обработки: {minutes} мин. {seconds:.2f} сек. ")
    print(f"Изображения сохранены в папку: {output_folder / f'{file_path.stem}_assets'}")
    print("-" * 60)


In [ ]:
OLLAMA_MODEL = "gemma4:31b-cloud"  # "gpt-oss:20b-cloud"

In [ ]:
def run_llama_extraction(markdown_text, model_name = "gemma4:31b-cloud"):
    """
    Запрос к облачному API Ollama.
    """
    start_time = time.time()
    print(f"[LlamaIndex] Подключение к облаку Ollama для модели {model_name}...")

    # Извлекаем API-ключ
    api_key = userdata.get('ocr_olama')
    if not api_key:
        raise ValueError("Добавьте секрет 'ocr_olama' с вашим Ollama API Key.")

    llm = OpenAILike(
        model=model_name,
        api_key=api_key,
        api_base="https://ollama.com/v1",  # Официальный облачный эндпоинт Ollama
        is_chat_model=True,
        temperature=0.1,
        max_tokens=2048,
        timeout=360.0
    )

    prompt = f"""
    Перед тобой структурированное представление документа в формате Markdown.
    Твоя задача — внимательно проанализировать его и составить краткий аналитический отчет/резюме на русском языке (1-2 абзаца).

    Сфокусируйся на:
    - Основной цели документа / исследования.
    - Ключевых выводах, результатах или метриках.

    Важные требования:
    - Пиши только связный текст отчета на русском языке.
    - Не добавляй вводных фраз ("Вот ваш отчет:", "Конечно, я помогу" и т.д.).

    Документ для анализа:
    ---
    {markdown_text}
    ---
    """

    print("[LlamaIndex] Отправка запроса в Ollama...")
    response = llm.complete(prompt)
    report = response.text.strip()

    end_time = time.time()
    processing_time = end_time - start_time
    print(f"[LlamaIndex] Анализ завершен за {processing_time:.2f} сек.")

    return report

In [ ]:
input_md_folder = Path("/content/drive/MyDrive/md_results2")
output_folder2 = Path("/content/drive/MyDrive/final_results")
output_folder2.mkdir(parents=True, exist_ok=True)


md_files = list(input_md_folder.glob("*.md"))
print(f"\nНайдено Markdown-файлов для анализа: {len(md_files)}")

for md_file_path in md_files:
    print(f"\nАнализ файла: {md_file_path.name}")
    print("-" * 60)

    try:
        with open(md_file_path, "r", encoding="utf-8") as f:
            markdown_content = f.read()

        # Создание отчета (использует модель из переменной OLLAMA_MODEL)
        extracted_report = run_llama_extraction(markdown_content, model_name=OLLAMA_MODEL)

        # Сохранение текстового отчета
        txt_output_path = output_folder2 / (md_file_path.stem + "_report.txt")
        with open(txt_output_path, "w", encoding="utf-8") as f:
            f.write(extracted_report)

        print(f"Текстовый отчет успешно сохранен: {txt_output_path}")
        print("\n=== КРАТКИЙ АНАЛИТИЧЕСКИЙ ОТЧЕТ ===")
        print(extracted_report)
        print("-" * 60)

    except Exception as e:
        print(f"[ОШИБКА] Не удалось обработать файл {md_file_path.name}: {e}")
        print("Переходим к следующему файлу...")
        print("-" * 60)
        continue

print("\nОбработка и генерация отчетов завершена.")